# Hugging Face Inference Endpoint


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


`langchain-huggingface`의 `HuggingFaceEndpoint`는 서버리스 Inference Providers와 전용 Endpoint를 연결합니다. 채팅 모델에는 특수 토큰을 직접 작성하지 않고 `ChatHuggingFace`가 토크나이저의 chat template을 적용하게 합니다.

`.env`에 `HUGGINGFACEHUB_API_TOKEN`을 설정하세요. 노트북의 `login()` 프롬프트나 하드코딩 토큰은 재현성과 보안 측면에서 피합니다.


In [ ]:
%pip install -qU langchain-huggingface huggingface_hub transformers python-dotenv


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    raise RuntimeError(".env에 HUGGINGFACEHUB_API_TOKEN을 설정하세요.")


## Serverless Inference Providers


In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

endpoint = HuggingFaceEndpoint(
    repo_id=os.getenv("HF_REPO_ID", "deepseek-ai/DeepSeek-R1-0528"),
    task="text-generation",
    provider=os.getenv("HF_INFERENCE_PROVIDER", "auto"),
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
)
chat = ChatHuggingFace(llm=endpoint)

response = chat.invoke(
    [
        ("system", "간결하고 정확하게 답하세요."),
        ("human", "대한민국의 수도는 어디인가요?"),
    ]
)
print(response.text)


## 전용 Endpoint

Endpoint URL은 비밀은 아니더라도 환경별 설정이므로 코드에 고정하지 않습니다. `HF_ENDPOINT_URL`로 주입합니다.


In [ ]:
endpoint_url = os.getenv("HF_ENDPOINT_URL")

if endpoint_url is None:
    print("전용 Endpoint 예제를 실행하려면 HF_ENDPOINT_URL을 설정하세요.")
else:
    dedicated_llm = HuggingFaceEndpoint(
        endpoint_url=endpoint_url,
        max_new_tokens=512,
        do_sample=False,
    )
    dedicated_chat = ChatHuggingFace(llm=dedicated_llm)
    result = dedicated_chat.invoke("RAG를 한 문장으로 설명해 주세요.")
    print(result.text)


### 선택 기준

- 빠른 실험: Inference Providers + `provider="auto"`
- 공급자 고정: `provider="together"`처럼 명시
- 예측 가능한 용량·네트워크 격리: 전용 Inference Endpoint
- 모델이 해당 공급자에서 실제 제공되는지는 실행 시점의 Hugging Face 모델 페이지에서 확인
